In [ ]:
import os
import requests
import csv
import pandas as pd
import time

API_KEY = os.getenv("GOOGLE_PLACES_API_KEY")

In [ ]:
PLACES_SEARCH_URL = "https://places.googleapis.com/v1/places:searchText"
PLACE_DETAILS_URL = "https://places.googleapis.com/v1/places/{place_id}"

In [ ]:
OUTPUT_FILE = "../datasets/university_reviews.csv"

In [ ]:
row_range = slice(400, 500)
unis = pd.read_csv("baseline_df_names_cleaned.csv", header=None, names=["name"])["name"].iloc[row_range]

In [ ]:
print(unis)

In [ ]:
with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    for uni in unis:
        try:
            # Find place
            search = requests.post(
                PLACES_SEARCH_URL,
                headers={
                    "Content-Type": "application/json",
                    "X-Goog-Api-Key": API_KEY,
                    "X-Goog-FieldMask": "places.id,places.displayName"
                },
                json={"textQuery": uni}
            ).json()

            if not search.get("places"):
                writer.writerow([uni, []])
                continue

            place_id = search["places"][0]["id"]
            matched_name = search["places"][0]["displayName"]["text"]

            # Get reviews
            details = requests.get(
                PLACE_DETAILS_URL.format(place_id=place_id),
                headers={
                    "Content-Type": "application/json",
                    "X-Goog-Api-Key": API_KEY,
                    "X-Goog-FieldMask": "reviews"
                }
            ).json()

            reviews = [r["text"]["text"] for r in details.get("reviews", []) if "text" in r]
            writer.writerow([uni, reviews])
            print(f"✓ {uni} ({len(reviews)} reviews)")

        except Exception as e:
            print(f"✗ {uni}: {e}")
            writer.writerow([uni, []])

        time.sleep(0.05)  # small delay to be safe
